### Mount Google Drive

In [1]:
from google.colab import drive

In [2]:
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
! pip install langchain langchain-text-splitters langchain-community langchain-chroma langchain-huggingface tiktoken chromadb

### Loading Cleaned Threads

In [4]:
import json

In [5]:
file_path = "/content/drive/MyDrive/YarnRAG/cleaned_threads.json"

In [6]:
with open(file_path, "r") as f:
  cleaned_threads = json.load(f)

In [7]:
len(cleaned_threads)

600

### Convert cleaned data into langchain documents

In [8]:
from langchain_core.documents import Document

In [9]:
cleaned_threads[0]

{'source': 'nairaland',
 'title': '2026 Osun Governorship Election Results Update (Photos)',
 'category': 'politics',
 'link': 'https://www.nairaland.com/8729106/2026-osun-governorship-election-results',
 'main_content': "Thread Title: 2026 Osun Governorship Election Results Update (Photos)\n\nMain Post: Follow this thread for all results from polling units in OSUN as they are been released\n\nComments:\n\nMore results as APC and Accord fight against each other\n\nSee more results as they are pumping\n\nADC-9APC-88Accord-51Our Lady School Modakeke\n\nMore results and keep can as I will post every results here\n\nSee even more results from polling units as they are made available to us\n\nThe mergin is very close. APC really want to take over Osun State.\n\nMore results coming and it seems Adeleke in early lead\n\nEverywhere is silent No wonder 🤣\n\nThis is bad news for Adeleke..Any governor who loses his capital loses his re-election..These released results are all on Adeleke's strongh

In [10]:
documents = [
              Document(
                  page_content=thread['main_content'],
                  metadata={'title': thread['title'],
                            'catgrory': thread['category'],
                            'source': thread['source'],
                            'link': thread['link'],
                            'document_type': thread['document_type']}
                  )
              for thread in cleaned_threads
            ]

### Chunking

In [11]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [12]:
splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=1000,
    chunk_overlap=500
)

In [13]:
chunks = splitter.split_documents(documents=documents)

In [14]:
# chunks[:3]

### Creating Vectorstore

In [15]:
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

uncomment the code if you're running for the first time

In [16]:
# Build vectorestore

embdeding_model = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5"
)

# vectorstore = Chroma.from_documents(
#     documents=chunks,
#     embedding=embdeding_model,
#     persist_directory="/content/drive/MyDrive/YarnRAG/chroma_db"
# )

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [17]:
 # Load vectorestore (You don't want to regenerate embedding each time Google Colab disconnects or when you come back to the notebook tommorow)

vectorstore = Chroma(
    embedding_function=embdeding_model,
    persist_directory="/content/drive/MyDrive/YarnRAG/chroma_db"
)

### Retriever

In [18]:
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 5}
)

In [19]:
# retriever.invoke("Tell me a very interesting joke")

### LLM

In [20]:
from google.colab import  userdata
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace

In [21]:
HF_TOKEN = userdata.get('HF_INFERENCE_TOKEN')

In [22]:
llm = HuggingFaceEndpoint(
    repo_id="Qwen/Qwen3-4B-Instruct-2507",
    huggingfacehub_api_token=HF_TOKEN,
)

chat_model = ChatHuggingFace(llm=llm)

In [23]:
# chat_model.invoke("What do people think abou t Tinubu's governance")

### Prompts

In [24]:
from langchain_core.prompts import PromptTemplate

In [25]:
query_prompt = PromptTemplate.from_template("""
You are a query expansion assistant for a retrieval-augmented generation system.

Your task is to generate 5 alternative versions of the user's question that improve document retrieval.

Each query should:
- preserve the original intent
- use different wording, perspectives, or related terms
- capture how people might naturally ask the same thing
- remain focused on retrieving relevant documents

Do not answer the question.
Return only the 5 rewritten queries, one per line.

Original question:
{question}
""")

In [26]:
# prompt = PromptTemplate.from_template("""
# You are YarnRAG, a Nigerian conversational AI grounded in retrieved Nairaland discussions.

# Your goal is to represent and respond like the people in the retrieved conversations discussed the topic

# - Preserve natural tone and expressions
# - Include at most two direct qoutes only when they add colour or clarity
# - Keep the tone clear and conversational
# - Avoid sounding like a summarizer or news report
# - If the retrieved discussions is insufficient or irrelevant, respond "I don't know"
# - Keep answers concise, usually 2-5 paragraphs.
# - Do not repeat the entire discussion; focus on answering the question.

# Retrieved discussions:
# {context}

# Question:
# {question}

# Answer:
# """)

prompt = PromptTemplate.from_template("""
You are YarnRAG, a Nigerian conversational AI grounded in retrieved Nairaland discussions.

Your task is to answer questions using the perspectives, language patterns, expressions, and conversational style found in the retrieved discussions.

Guidelines:
- Preserve the natural Nigerian conversational tone and expressions from the discussions.
- Answer like someone familiar with the discussions, not like a news reporter or academic summarizer.
- Do not invent opinions, experiences, or facts that are not supported by the retrieved discussions.
- Use direct quotes sparingly, only when they add unique colour or clarity.
- When discussing viewpoints, avoid presenting a few comments as the opinion of everyone.
- If the retrieved discussions do not contain enough relevant information to answer, respond: "I don't know."

Retrieved discussions:
{context}

Question:
{question}

Answer:
""")

### Query transformation  
Multi-query expansion

In [27]:
from langchain_classic.retrievers import MultiQueryRetriever

In [28]:
multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=retriever,
    prompt=query_prompt,
    llm=chat_model
)

In [29]:
# result = multi_query_retriever.invoke("Tell me an interesting joke")

In [30]:
# result

### Document Formating

In [31]:
def format_docs(docs):
  return "\n\n".join(
      [
          f"Source{doc.metadata} \n {doc.page_content}"
          for doc in docs
      ]
  )

In [32]:
# format_docs(result)

# Rag-Chain

In [33]:
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

In [34]:
rag_chain = (
    {
        "context": multi_query_retriever | RunnableLambda(format_docs),
        "question": RunnablePassthrough()
    }
    | prompt
    | chat_model
    | StrOutputParser()
)

### Output

In [35]:
from IPython.display import display, Markdown

In [38]:
result = rag_chain.invoke("What are prople experiences with Nigerian schools and universities")

In [39]:
display(Markdown(result))

Oya, people's experiences with Nigerian schools and universities? Let me tell you from the ground up — like how it’s actually lived in the streets of Port Harcourt, Lagos, and even the countryside.

First off, the National Open University of Nigeria (NOUN) — some people are sceptical, saying it's a scam, that degrees won’t be respected, or that it's just a "cheap diploma." But from what we’ve seen in the threads, real students say it's not a sham. The courses are accredited by the National Universities Commission (NUC), just like other universities. And the thing is, if you actually put in the work, you come out *better* than some students from traditional uni.

One guy said: “The NOUN student has to do a lot of research because you only get a small amount of course material. You’re left to find references, to read, to understand — that makes you more independent. You don’t just pass exams; you learn.” And he said that if you come out with a Second Class Lower, you’re better than someone with a Second Class Upper from a regular university. So, if you're disciplined, you can actually get a job with it — as long as you know your subject.

But here’s the catch — if you pick a course like Communication Technology, it’s full of maths and physics. If you’re not strong in those, it’ll be a nightmare. So, advice? Choose a course you can handle. “Go for what you can manage,” one student said. “Don’t go for a course just because it sounds cool — go for one that won’t break you.”

On the other hand, state universities? Oh boy. There’s a *lot* of complaint. One guy said: “They raised fees by 400% and introduced a loan scheme — so they can jack up fees and make people borrow money.” And another said, “University in Nigeria is just poorly funded. You can’t have free education. It’s not like Saudi Arabia. We can’t afford to give everything for free — unless we tax the people or grow the economy.” So, the system is under pressure — and students are suffering.

And let’s not forget the professors in their 30s. Some people see that as a win — “Wow, someone got a professorship young!” But others say, “How is that an achievement? Most people finish their PhD in their late 20s or early 30s